In [1]:
import pandas as pd
import re
import os
from fuzzywuzzy import fuzz
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from tqdm import tqdm
from datetime import datetime

In [2]:
# === CONFIG ===

# Input study file + sheet
STUDY_FILE = './out/HDP01233_combined_DD_2026-04-16_matches confirmed.xlsx'
STUDY_SHEET = 'EnhancedDD'

# Study variable columns
ENCODING_COLUMN = 'Choices, Calculations, OR Slider Labels'
FIELD_LABEL_COLUMN = 'Field Label'
VARIABLE_NAME_COLUMN = 'Variable / Field Name'
FORM_NAME_COLUMN = 'Form Name'   # new: useful for concept retrieval context

# HEAL CDE knowledge base
CDE_FILE = './KnowledgeBase/Compiled_CORE_CDEs list_English_one sheet_as of 2025-01-28.xlsx'

# --- Deterministic output directory ---
# Works in BOTH notebooks and .py scripts
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # __file__ is not defined in notebooks / interactive sessions
    BASE_DIR = os.getcwd()

OUTPUT_DIR = os.path.join(BASE_DIR, "out")
print("📁 Output directory (absolute):", os.path.abspath(OUTPUT_DIR))

# --- CRF-aware concept retrieval settings ---
EXPECTED_CRF_COL = "HEAL Core CRF Match"
PREFER_EXPECTED_CRF_FOR_CONCEPT = True   # softer than "restrict"
CRF_MAP_THRESHOLD = 85
CRF_CONCEPT_BONUS = 8                    # small bonus for candidates inside expected CRF

# --- Stage 1: concept retrieval thresholds ---
CONCEPT_THRESHOLDS = {
    'high': 75,
    'medium': 50,
    'minimum_score': 25
}

# --- Stage 2: implementation fidelity thresholds ---
FIDELITY_THRESHOLDS = {
    'high': 80,
    'medium': 50
}

# --- Top N concept candidates to retain ---
TOP_N_CONCEPT_CANDIDATES = 3

📁 Output directory (absolute): c:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out


In [3]:
# ============================================================
# STAGE 1 + STAGE 2 SCORING HELPERS
# Add this as a NEW cell below the big helper/matching cell.
# These later definitions will override earlier ones in memory.
# ============================================================

# We no longer use one combined score as the primary decision maker.
# Instead:
#   - concept_similarity_score(...) chooses the closest HEAL CDE concept
#   - encoding_fidelity_score(...) describes how similarly it was implemented

VAR_NAME_WEIGHT = 0.25
FIELD_LABEL_WEIGHT = 0.50
FORM_CONTEXT_WEIGHT = 0.15
CRF_CONTEXT_WEIGHT = 0.10

def normalize_string(s):
    """Normalize strings for fuzzy concept/fidelity comparison."""
    if pd.isna(s) or s is None:
        return ""
    s = str(s).lower().strip()
    s = re.sub(r'[^a-zA-Z0-9\s=]', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

def similarity_score(str1, str2):
    """Token-set fuzzy similarity, 0-100."""
    str1 = normalize_string(str1)
    str2 = normalize_string(str2)

    if not str1 and not str2:
        return 0
    if not str1 or not str2:
        return 0

    return fuzz.token_set_ratio(str1, str2)

# --- CRF helpers (redefined here so downstream logic still works) ---
def normalize_crf_label(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    junk = {"questionnaire", "scale", "inventory", "form", "survey", "assessment"}
    tokens = [t for t in s.split() if t not in junk]
    return " ".join(tokens)

def build_crf_mapper(kb_crf_names):
    """
    Returns a function that maps arbitrary CRF labels to the closest KB CRF Name.
    """
    kb = [str(x).strip() for x in kb_crf_names if pd.notna(x) and str(x).strip()]
    kb_norm = {name: normalize_crf_label(name) for name in kb}

    def map_crf(label):
        if not isinstance(label, str) or not label.strip():
            return None

        norm = normalize_crf_label(label)

        # exact normalized hit
        for kb_name, kb_norm_name in kb_norm.items():
            if norm == kb_norm_name:
                return kb_name

        # fuzzy fallback
        best_name = None
        best_score = -1
        for kb_name, kb_norm_name in kb_norm.items():
            score = fuzz.token_set_ratio(norm, kb_norm_name)
            if score > best_score:
                best_name = kb_name
                best_score = score

        return best_name if best_score >= CRF_MAP_THRESHOLD else None

    return map_crf

# --- Stage 1: concept retrieval ---
def concept_similarity_score(
    study_var_name,
    study_field_label,
    study_form_name,
    expected_crf,
    cde_var_name,
    cde_question_text,
    cde_crf_name
):
    """
    Score conceptual closeness between a study row and a HEAL CDE candidate.
    This is the PRIMARY selector for the closest concept.
    Encoding is intentionally excluded here.
    """
    var_score = similarity_score(study_var_name, cde_var_name)
    label_score = similarity_score(study_field_label, cde_question_text)
    form_score = similarity_score(study_form_name, cde_crf_name)

    score = (
        var_score * VAR_NAME_WEIGHT +
        label_score * FIELD_LABEL_WEIGHT +
        form_score * FORM_CONTEXT_WEIGHT
    )

    # Small bonus if candidate lives inside the expected CRF context
    if PREFER_EXPECTED_CRF_FOR_CONCEPT and expected_crf and cde_crf_name == expected_crf:
        score += CRF_CONCEPT_BONUS

    return min(score, 100)

# --- Stage 2: implementation fidelity ---
def encoding_fidelity_score(study_encoding, cde_encoding):
    """
    Score how similarly the concept appears to be implemented.
    This does NOT decide the concept winner. It describes implementation closeness.
    """
    study_encoding = normalize_string(study_encoding)
    cde_encoding = normalize_string(cde_encoding)

    if not study_encoding and not cde_encoding:
        return 0
    if not study_encoding or not cde_encoding:
        return 0

    return similarity_score(study_encoding, cde_encoding)

def classify_concept_score(score):
    if pd.isna(score):
        return "No score"
    if score >= CONCEPT_THRESHOLDS['high']:
        return "High concept match"
    elif score >= CONCEPT_THRESHOLDS['medium']:
        return "Possible concept match"
    elif score >= CONCEPT_THRESHOLDS['minimum_score']:
        return "Weak concept match"
    return "No confident concept match"

def classify_fidelity_score(score):
    if pd.isna(score):
        return "No score"
    if score >= FIDELITY_THRESHOLDS['high']:
        return "Closely implemented"
    elif score >= FIDELITY_THRESHOLDS['medium']:
        return "Concept captured, encoding differs"
    return "Low implementation fidelity"

print("✅ Stage 1/Stage 2 scoring helpers loaded.")
print("   - Concept similarity now drives candidate selection")
print("   - Encoding fidelity is now a separate descriptive score")

✅ Stage 1/Stage 2 scoring helpers loaded.
   - Concept similarity now drives candidate selection
   - Encoding fidelity is now a separate descriptive score


In [4]:
# ============================================================
# SELECTIVE STAGE 1 CONCEPT-FAMILY VIEW
# Add this as a NEW cell below the Stage 1/Stage 2 helper cell
# ============================================================

# Only collapse known parent-concept families here.
# Start tiny and safe.
CONCEPT_FAMILY_WHITELIST = {
    ("Demographics", "Race"),
}

def build_stage1_concept_view(cde_df):
    """
    Build a Stage 1 concept-retrieval table.

    Behavior:
    - For whitelisted parent concepts (e.g., Demographics | Race),
      collapse child rows into ONE synthetic concept row.
    - For everything else, keep the original KB rows as-is.

    Returns a dataframe that still has the same key columns used downstream:
    - CRF Name
    - CDE Name
    - Variable Name
    - Definition
    - Short Description
    - Additional Notes (Question Text)
    - PV Description
    """
    rows = []

    # group by CRF + CDE Name so we can selectively collapse
    grouped = cde_df.groupby(["CRF Name", "CDE Name"], dropna=False)

    for (crf_name, cde_name), grp in grouped:
        key = (str(crf_name).strip(), str(cde_name).strip())

        # --- FAMILY MODE: collapse to one synthetic parent concept ---
        if key in CONCEPT_FAMILY_WHITELIST:
            member_vars = sorted({
                str(v).strip()
                for v in grp["Variable Name"].dropna().tolist()
                if str(v).strip()
            })

            definitions = sorted({
                str(v).strip()
                for v in grp["Definition"].dropna().tolist()
                if str(v).strip()
            })

            short_descs = sorted({
                str(v).strip()
                for v in grp["Short Description"].dropna().tolist()
                if str(v).strip()
            })

            question_texts = sorted({
                str(v).strip()
                for v in grp["Additional Notes (Question Text)"].dropna().tolist()
                if str(v).strip()
            })

            family_question_text = " | ".join([
                f"CDE Name: {cde_name}",
                f"Member variables: {', '.join(member_vars)}" if member_vars else "",
                f"Definitions: {' ; '.join(definitions[:2])}" if definitions else "",
                f"Question text: {' ; '.join(question_texts[:2])}" if question_texts else ""
            ]).strip(" |")

            rows.append({
                "CRF Name": crf_name,
                "CDE Name": cde_name,
                # make the synthetic parent variable name the concept label
                "Variable Name": cde_name,
                "Definition": " ; ".join(definitions),
                "Short Description": " ; ".join(short_descs),
                "Additional Notes (Question Text)": family_question_text,
                "PV Description": "",
                "Stage1 Concept Mode": "family",
                "Stage1 Member Variables": ", ".join(member_vars)
            })

        # --- ROW MODE: keep original KB rows ---
        else:
            for _, r in grp.iterrows():
                rows.append({
                    "CRF Name": r.get("CRF Name", ""),
                    "CDE Name": r.get("CDE Name", ""),
                    "Variable Name": r.get("Variable Name", ""),
                    "Definition": r.get("Definition", ""),
                    "Short Description": r.get("Short Description", ""),
                    "Additional Notes (Question Text)": r.get("Additional Notes (Question Text)", ""),
                    "PV Description": r.get("PV Description", ""),
                    "Stage1 Concept Mode": "row",
                    "Stage1 Member Variables": ""
                })

    concept_df = pd.DataFrame(rows)

    print("✅ Stage 1 concept view created.")
    print(f"   Original KB rows: {len(cde_df)}")
    print(f"   Stage 1 concept rows: {len(concept_df)}")

    family_rows = concept_df[concept_df["Stage1 Concept Mode"] == "family"]
    if not family_rows.empty:
        print("\n🧩 Family-mode concepts currently enabled:")
        display(family_rows[["CRF Name", "CDE Name", "Variable Name", "Stage1 Member Variables"]])
    else:
        print("\n🧩 No family-mode concepts enabled yet.")

    return concept_df

In [5]:
def main(dry_run=False):
    """Main function to run the VLMD CDE matching process."""

    print("🚀 Starting HEAL CDE Variable Level Metadata Matching")
    print(
        f"📋 Concept retrieval thresholds: High ≥{CONCEPT_THRESHOLDS['high']}%, "
        f"Possible ≥{CONCEPT_THRESHOLDS['medium']}%, "
        f"Minimum ≥{CONCEPT_THRESHOLDS['minimum_score']}%"
    )
    print(
        f"📋 Encoding fidelity thresholds: High ≥{FIDELITY_THRESHOLDS['high']}%, "
        f"Medium ≥{FIDELITY_THRESHOLDS['medium']}%"
    )

    print("\n🧭 Concept retrieval settings:")
    print(f"   Expected CRF column        : {EXPECTED_CRF_COL}")
    print(f"   Prefer expected CRF        : {PREFER_EXPECTED_CRF_FOR_CONCEPT}")
    print(f"   CRF mapping threshold      : {CRF_MAP_THRESHOLD}")
    print(f"   CRF concept bonus          : {CRF_CONCEPT_BONUS}")
    print("   (Potential Match 2/3 are alternate concept candidates.)")

    try:
        preview_df = pd.read_excel(STUDY_FILE, sheet_name=STUDY_SHEET, nrows=5)
        if EXPECTED_CRF_COL not in preview_df.columns:
            print(
                f"\n⚠️ Warning: Expected CRF column '{EXPECTED_CRF_COL}' not found in '{STUDY_SHEET}'.\n"
                "   Concept retrieval will continue without CRF preference.\n"
            )
    except Exception as e:
        print(f"\n⚠️ Preflight check skipped (could not read file): {e}\n")

    output_file = compare_encodings(
        STUDY_FILE,
        encoding_column=ENCODING_COLUMN,
        field_label_column=FIELD_LABEL_COLUMN,
        cde_file=CDE_FILE,
        study_sheet=STUDY_SHEET,
        variable_name_column=VARIABLE_NAME_COLUMN,
        dry_run=dry_run
    )

    if dry_run:
        print("\n🧪 Dry run complete (no output file written).")
        return

    if not output_file:
        print("\n❌ No output file was returned. Check logs above for any errors.")
        return

    print("\n✅ Output file created:")
    print("   📄", os.path.abspath(output_file))

    try:
        apply_color_coding(output_file)
    except Exception as e:
        print(f"\n⚠️ Color coding failed (file is still valid): {e}")

    print("\n🎉 VLMD CDE matching complete!")
    print(f"📊 Results saved to: {output_file}")
    print("📋 Review concept matches, encoding fidelity, and the Low_Confidence_Analysis sheet.")

In [6]:
# ============================================================
# STAGE A DISPLAY POLICY HELPERS
# Add this as a NEW cell above the override compare_encodings() cell
# ============================================================

def has_usable_crf_anchor(raw_crf_value):
    """
    True if the row has a real HEAL Core CRF Match we can use as an anchor.
    """
    if pd.isna(raw_crf_value):
        return False

    value = str(raw_crf_value).strip()
    if value == "" or value.lower() == "no crf match":
        return False

    return True


def classify_stage_a_display_status(concept_score, has_anchor, expected_kb=None, best_crf_name=None, retrieval_mode=None):
    """
    Final display status policy for Stage A.

    Rules:
    - Anchored rows are only allowed to be High if:
        1) score is high
        2) winner stayed inside the expected CRF lane
        3) retrieval came from anchored_crf_first
    - Anchored rows otherwise fall back to Possible / Weak
    - Unanchored rows are capped at Possible
    """
    if pd.isna(concept_score):
        return "Weak concept match"

    score = float(concept_score)

    if has_anchor:
        stayed_in_anchor_lane = (
            expected_kb is not None and
            best_crf_name is not None and
            str(best_crf_name).strip() == str(expected_kb).strip() and
            retrieval_mode == "anchored_crf_first"
        )

        if score >= CONCEPT_THRESHOLDS['high'] and stayed_in_anchor_lane:
            return "High concept match"
        elif score >= CONCEPT_THRESHOLDS['medium']:
            return "Possible concept match"
        else:
            return "Weak concept match"
    else:
        # Unanchored rows can never be High
        if score >= CONCEPT_THRESHOLDS['medium']:
            return "Possible concept match"
        else:
            return "Weak concept match"


def should_blank_main_concept(display_status):
    """
    If True, we will later blank out the main concept columns in VLMD_Results.
    """
    return str(display_status).strip() == "Weak concept match"


print("✅ Stage A display policy helpers loaded.")
print("   - usable CRF anchor detector")
print("   - anchored rows eligible for High")
print("   - unanchored rows capped at Possible")
print("   - weak rows marked for blanking later")

✅ Stage A display policy helpers loaded.
   - usable CRF anchor detector
   - anchored rows eligible for High
   - unanchored rows capped at Possible
   - weak rows marked for blanking later


In [7]:
# ============================================================
# COPYRIGHT-SENSITIVE FAMILY RULES
# Add this as a NEW cell above compare_encodings(...)
# ============================================================

def normalize_crf_label(value):
    if pd.isna(value) or value is None:
        return ""
    return str(value).strip().lower()

# Conservative first-pass rules:
# If the expected CRF is in one of these protected groups,
# only allow primary best-match assignment from the same allowed family.
# Cross-family "proxy" candidates can still be shown, but NOT accepted as Best Match.
PROTECTED_PRIMARY_FAMILY_RULES = {
    "brief pain inventory (bpi)": {
        "brief pain inventory (bpi)",
        "bpi pain severity",
        "bpi pain interference"
    },
    "bpi pain interference": {
        "brief pain inventory (bpi)",
        "bpi pain interference"
    },
    "bpi pain severity": {
        "brief pain inventory (bpi)",
        "bpi pain severity"
    },
    "pcs-6": {
        "pcs-6"
    },
    "pcs-13": {
        "pcs-13"
    },
    "pedsql inventory": {
        "pedsql inventory",
        "pedsql (pediatric quality of life inventory)",
        "pedsql"
    },
    "pedsql (pediatric quality of life inventory)": {
        "pedsql inventory",
        "pedsql (pediatric quality of life inventory)",
        "pedsql"
    },
    "pedsql": {
        "pedsql inventory",
        "pedsql (pediatric quality of life inventory)",
        "pedsql"
    }
}

def is_copyright_sensitive_expected_crf(expected_kb):
    """
    True if the expected HEAL Core CRF belongs to a protected family.
    """
    return normalize_crf_label(expected_kb) in PROTECTED_PRIMARY_FAMILY_RULES

def is_blocked_primary_candidate(expected_kb, candidate_crf):
    """
    For protected families only:
    - block cross-family candidates from becoming the primary best match
    - still allow them to exist as reviewable proxy suggestions
    """
    expected_norm = normalize_crf_label(expected_kb)
    candidate_norm = normalize_crf_label(candidate_crf)

    if expected_norm not in PROTECTED_PRIMARY_FAMILY_RULES:
        return False

    allowed_family = PROTECTED_PRIMARY_FAMILY_RULES[expected_norm]
    return candidate_norm not in allowed_family

print("✅ Copyright-sensitive family helpers loaded.")
print("Protected CRF families:", list(PROTECTED_PRIMARY_FAMILY_RULES.keys()))

✅ Copyright-sensitive family helpers loaded.
Protected CRF families: ['brief pain inventory (bpi)', 'bpi pain interference', 'bpi pain severity', 'pcs-6', 'pcs-13', 'pedsql inventory', 'pedsql (pediatric quality of life inventory)', 'pedsql']


In [8]:
# ============================================================
# OVERRIDE SUMMARY + MATCHING LOGIC
# Replace your entire current override cell with this block.
# ============================================================

# Compatibility aliases so the old main()/console text still works
CONFIDENCE_THRESHOLDS = CONCEPT_THRESHOLDS
RESTRICT_TO_EXPECTED_CRF_FIRST = PREFER_EXPECTED_CRF_FOR_CONCEPT
ALLOW_GLOBAL_FALLBACK_FOR_BEST_MATCH = True


def print_matching_summary(final_df):
    """Updated summary using the new clearer final-match columns."""
    processed_df = final_df[final_df['Final HEAL CDE Concept Match'].notna()].copy()
    total_processed = len(processed_df)

    if total_processed == 0:
        print("📊 No concept matches found.")
        return

    high_concept = len(processed_df[processed_df['Final Concept Match Score'] >= CONCEPT_THRESHOLDS['high']])
    medium_concept = len(processed_df[
        (processed_df['Final Concept Match Score'] >= CONCEPT_THRESHOLDS['medium']) &
        (processed_df['Final Concept Match Score'] < CONCEPT_THRESHOLDS['high'])
    ])
    low_concept = len(processed_df[processed_df['Final Concept Match Score'] < CONCEPT_THRESHOLDS['medium']])

    family_mode_count = len(processed_df[processed_df['Final Concept Match Mode'] == 'family'])

    fidelity_df = processed_df[processed_df['Final Encoding Fidelity Score'].notna()].copy()
    total_fidelity = len(fidelity_df)

    print(f"\n📊 Matching Summary:")
    print(f"   Total variables with a final concept match: {total_processed}")
    print(f"   🧠 High concept matches (≥{CONCEPT_THRESHOLDS['high']}): {high_concept} ({high_concept/total_processed*100:.1f}%)")
    print(f"   🧠 Possible concept matches ({CONCEPT_THRESHOLDS['medium']}-{CONCEPT_THRESHOLDS['high']-1}): {medium_concept} ({medium_concept/total_processed*100:.1f}%)")
    print(f"   🧠 Weak concept matches (<{CONCEPT_THRESHOLDS['medium']}): {low_concept} ({low_concept/total_processed*100:.1f}%)")
    print(f"   🧩 Family-level concept matches: {family_mode_count}")

    if total_fidelity > 0:
        high_fidelity = len(fidelity_df[fidelity_df['Final Encoding Fidelity Score'] >= FIDELITY_THRESHOLDS['high']])
        medium_fidelity = len(fidelity_df[
            (fidelity_df['Final Encoding Fidelity Score'] >= FIDELITY_THRESHOLDS['medium']) &
            (fidelity_df['Final Encoding Fidelity Score'] < FIDELITY_THRESHOLDS['high'])
        ])
        low_fidelity = len(fidelity_df[fidelity_df['Final Encoding Fidelity Score'] < FIDELITY_THRESHOLDS['medium']])

        print(f"\n   🛠️ High implementation fidelity (≥{FIDELITY_THRESHOLDS['high']}): {high_fidelity} ({high_fidelity/total_fidelity*100:.1f}%)")
        print(f"   🛠️ Medium implementation fidelity ({FIDELITY_THRESHOLDS['medium']}-{FIDELITY_THRESHOLDS['high']-1}): {medium_fidelity} ({medium_fidelity/total_fidelity*100:.1f}%)")
        print(f"   🛠️ Low implementation fidelity (<{FIDELITY_THRESHOLDS['medium']}): {low_fidelity} ({low_fidelity/total_fidelity*100:.1f}%)")
    else:
        print("\n   🛠️ No row-level implementation fidelity scores were available.")


def compare_encodings(
    study_file,
    encoding_column='encodings',
    field_label_column='field_label',
    cde_file='./KnowledgeBase/Compiled_CORE_CDEs list_English_one sheet_as of 2025-01-28.xlsx',
    study_sheet='Sheet1',
    variable_name_column='name',
    dry_run=False
):
    """
    New behavior:
    - Stage 1 selects the winner by CONCEPT score
    - Stage 1 uses a selective concept-family view of the KB
    - Stage 2 describes implementation using ENCODING FIDELITY score
    - Clearer FINAL output columns are written
    - Legacy columns are still populated for compatibility
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print("📁 Output directory (absolute):", os.path.abspath(OUTPUT_DIR))

    # ----------------------------
    # Load study data
    # ----------------------------
    print("📂 Loading study data...")
    if study_file.endswith('.xlsx'):
        full_study_df = pd.read_excel(study_file, sheet_name=study_sheet)
    else:
        full_study_df = pd.read_csv(study_file)

    study_df = full_study_df.copy()

    if EXPECTED_CRF_COL in study_df.columns:
        no_crf_count = (study_df[EXPECTED_CRF_COL] == 'No CRF match').sum()
        print(f"✅ Processing all {len(study_df)} rows, including {no_crf_count} rows with 'No CRF match'.")
    else:
        print(f"✅ Processing all {len(study_df)} rows (no '{EXPECTED_CRF_COL}' column found).")

    if dry_run:
        print(f"🔍 DRY RUN: Would process {len(study_df)} variables against HEAL CDEs")
        return None

    # ----------------------------
    # Resolve study columns from config + fallback
    # ----------------------------
    resolved_cols = resolve_study_columns(study_df)

    var_col = resolved_cols["variable_name"]
    field_label_column = resolved_cols["field_label"]
    encoding_column = resolved_cols["encoding"]
    form_col = resolved_cols["form_name"]

    # expected CRF can be optional
    expected_crf_col = resolved_cols["expected_crf"]
    if expected_crf_col:
        print(f"🧭 Using '{expected_crf_col}' as the expected HEAL Core CRF column.")
    else:
        print("⚠️ No expected HEAL Core CRF column found. Concept retrieval will proceed without CRF preference.")

    if var_col:
        print(f"🔎 Using '{var_col}' as the study variable-name column.")
    else:
        print("⚠️ Could not find a variable-name column.")

    if form_col:
        print(f"🧾 Using '{form_col}' as the study form-name column.")
    else:
        print("⚠️ Could not find a form-name column. Form context score will be weaker.")

    # ----------------------------
    # Initialize output columns
    # ----------------------------
    new_cols = [
        # new clean-workflow columns
        'Closest HEAL CDE Concept',
        'Concept Match Score',
        'Concept Match CRF',
        'Concept Match Status',
        'Encoding Fidelity Score',
        'Encoding Fidelity Status',

        # legacy compatibility columns
        'Best Match CDE Name',
        'Best Match Score',
        'Best Match CRF Name',
        'Best Match Source',

        # top concept alternates
        'Potential Match 2 - CDE Name',
        'Potential Match 2 - Score',
        'Potential Match 2 - CRF Name',
        'Potential Match 3 - CDE Name',
        'Potential Match 3 - Score',
        'Potential Match 3 - CRF Name',

        # new copyright-sensitive / proxy-block columns
        'Protected Family Rule Applied',
        'Protected Family Expected CRF',
        'Blocked Primary Candidate',
        'Blocked Primary Candidate Score',
        'Blocked Primary Candidate CRF',
        'Blocked Primary Candidate Fidelity',
        'Protected Family Note'
    ]

    for col in new_cols:
        if col not in study_df.columns:
            study_df[col] = None
        else:
            study_df[col] = None

    # ----------------------------
    # Normalize study fields
    # ----------------------------
    print("🔧 Normalizing study variables...")
    study_df['Normalized Text'] = study_df[field_label_column].apply(normalize_string)
    study_df['Normalized Encoding'] = study_df[encoding_column].apply(normalize_string)

    if var_col:
        study_df['Normalized Variable Name'] = study_df[var_col].apply(normalize_string)
    else:
        study_df['Normalized Variable Name'] = ""

    if form_col:
        study_df['Normalized Form Name'] = study_df[form_col].apply(normalize_string)
    else:
        study_df['Normalized Form Name'] = ""

    study_df['Normalized Combined'] = study_df.apply(
        lambda row: normalize_string(
            f"{row.get(var_col, '')} | {row.get(field_label_column, '')} | {row.get(encoding_column, '')}"
        ),
        axis=1
    )

    # ----------------------------
    # Load KB + build Stage 1 concept view
    # ----------------------------
    print("📚 Loading HEAL CDE database...")
    cde_df = pd.read_excel(cde_file, sheet_name='ALL')

    cde_df = cde_df.dropna(subset=['CDE Name', 'Variable Name', 'Additional Notes (Question Text)'], how='all').copy()

    cde_df["CRF Name"] = cde_df["CRF Name"].astype(str).str.strip()
    cde_df["CDE Name"] = cde_df["CDE Name"].astype(str).str.strip()

    kb_crf_names = sorted(cde_df["CRF Name"].dropna().unique())
    map_crf = build_crf_mapper(kb_crf_names)

    cde_df['Normalized CDE Variable Name'] = cde_df['Variable Name'].apply(normalize_string)
    cde_df['Normalized Text'] = cde_df['Additional Notes (Question Text)'].apply(normalize_string)
    cde_df['Normalized Encoding'] = cde_df['PV Description'].apply(normalize_string)

    stage1_cde_df = build_stage1_concept_view(cde_df).copy()
    stage1_cde_df['Normalized CDE Variable Name'] = stage1_cde_df['Variable Name'].apply(normalize_string)
    stage1_cde_df['Normalized Text'] = stage1_cde_df['Additional Notes (Question Text)'].apply(normalize_string)
    stage1_cde_df['Normalized Encoding'] = stage1_cde_df['PV Description'].apply(normalize_string)

    print(f"✅ Loaded {len(cde_df)} raw KB rows for implementation detail.")
    print(f"✅ Built {len(stage1_cde_df)} Stage 1 concept rows for concept retrieval.")

    # ----------------------------
    # Track lower-confidence concept matches
    # ----------------------------
    low_confidence_matches = []

    # ----------------------------
    # Matching
    # ----------------------------
    print("🔍 Running concept retrieval first, then encoding fidelity scoring...")
    for idx, row in tqdm(study_df.iterrows(), total=len(study_df), desc="Matching variables", unit="vars"):

        if not row.get('Normalized Variable Name', '') and not row.get('Normalized Text', ''):
            continue

        expected_raw = row.get(expected_crf_col, None) if expected_crf_col else None
        has_anchor = has_usable_crf_anchor(expected_raw)

        if has_anchor:
            expected_kb = map_crf(expected_raw)
        else:
            expected_kb = None

        study_var_name = row.get(var_col, '') if var_col else ''
        study_field_label = row.get(field_label_column, '')
        study_form_name = row.get(form_col, '') if form_col else ''
        study_encoding = row.get(encoding_column, '')

        candidates = []

        for _, cde_row in cde_df.iterrows():
            cde_var = cde_row.get('Variable Name', '')
            cde_text = cde_row.get('Additional Notes (Question Text)', '')
            cde_encoding = cde_row.get('PV Description', '')
            cde_crf = cde_row.get('CRF Name', '')

            concept_score = concept_similarity_score(
                study_var_name=study_var_name,
                study_field_label=study_field_label,
                study_form_name=study_form_name,
                expected_crf=expected_kb,
                cde_var_name=cde_var,
                cde_question_text=cde_text,
                cde_crf_name=cde_crf
            )

            if concept_score < CONCEPT_THRESHOLDS['minimum_score']:
                continue

            fidelity_score = encoding_fidelity_score(study_encoding, cde_encoding)
            blocked_for_primary = is_blocked_primary_candidate(expected_kb, cde_crf)

            candidates.append({
                'cde_name': cde_var,
                'concept_score': round(concept_score, 1),
                'crf_name': cde_crf,
                'fidelity_score': round(fidelity_score, 1),
                'preferred_crf_hit': (expected_kb is not None and cde_crf == expected_kb),
                'blocked_for_primary': blocked_for_primary
            })

        if not candidates:
            continue

        # Sort by concept score first, then prefer expected CRF, then fidelity
        candidates = sorted(
            candidates,
            key=lambda x: (
                x['concept_score'],
                1 if x['preferred_crf_hit'] else 0,
                x['fidelity_score']
            ),
            reverse=True
        )

        # De-dupe by CDE name
        unique_candidates = []
        seen = set()
        for cand in candidates:
            if cand['cde_name'] in seen:
                continue
            unique_candidates.append(cand)
            seen.add(cand['cde_name'])

        protected_mode = is_copyright_sensitive_expected_crf(expected_kb)

        if protected_mode:
            study_df.at[idx, 'Protected Family Rule Applied'] = "Yes"
            study_df.at[idx, 'Protected Family Expected CRF'] = expected_kb
        else:
            study_df.at[idx, 'Protected Family Rule Applied'] = ""
            study_df.at[idx, 'Protected Family Expected CRF'] = ""

        allowed_primary_candidates = [cand for cand in unique_candidates if not cand['blocked_for_primary']]
        blocked_primary_candidates = [cand for cand in unique_candidates if cand['blocked_for_primary']]

        best = allowed_primary_candidates[0] if allowed_primary_candidates else None

        # ============================================================
        # CASE 1: normal / allowed primary winner exists
        # ============================================================
        if best is not None:
            top_candidates = allowed_primary_candidates[:TOP_N_CONCEPT_CANDIDATES]

            # New clean columns
            study_df.at[idx, 'Closest HEAL CDE Concept'] = best['cde_name']
            study_df.at[idx, 'Concept Match Score'] = best['concept_score']
            study_df.at[idx, 'Concept Match CRF'] = best['crf_name']
            study_df.at[idx, 'Concept Match Status'] = classify_concept_score(best['concept_score'])
            study_df.at[idx, 'Encoding Fidelity Score'] = best['fidelity_score']
            study_df.at[idx, 'Encoding Fidelity Status'] = classify_fidelity_score(best['fidelity_score'])

            # Legacy compatibility columns
            study_df.at[idx, 'Best Match CDE Name'] = best['cde_name']
            study_df.at[idx, 'Best Match Score'] = best['concept_score']
            study_df.at[idx, 'Best Match CRF Name'] = best['crf_name']
            study_df.at[idx, 'Best Match Source'] = "Concept retrieval"

            # Reset blocked-proxy columns for clean rows
            study_df.at[idx, 'Blocked Primary Candidate'] = None
            study_df.at[idx, 'Blocked Primary Candidate Score'] = None
            study_df.at[idx, 'Blocked Primary Candidate CRF'] = None
            study_df.at[idx, 'Blocked Primary Candidate Fidelity'] = None
            study_df.at[idx, 'Protected Family Note'] = None

            # Top alternates from allowed candidates only
            if len(top_candidates) > 1:
                study_df.at[idx, 'Potential Match 2 - CDE Name'] = top_candidates[1]['cde_name']
                study_df.at[idx, 'Potential Match 2 - Score'] = top_candidates[1]['concept_score']
                study_df.at[idx, 'Potential Match 2 - CRF Name'] = top_candidates[1]['crf_name']

            if len(top_candidates) > 2:
                study_df.at[idx, 'Potential Match 3 - CDE Name'] = top_candidates[2]['cde_name']
                study_df.at[idx, 'Potential Match 3 - Score'] = top_candidates[2]['concept_score']
                study_df.at[idx, 'Potential Match 3 - CRF Name'] = top_candidates[2]['crf_name']

            # Review sheet for weaker allowed primary matches
            if best['concept_score'] < CONCEPT_THRESHOLDS['high']:
                study_text_raw = f"{study_var_name} | {study_field_label} | {study_encoding}".strip()
                if len(study_text_raw) > 140:
                    study_text_raw = study_text_raw[:137] + "..."

                low_confidence_matches.append({
                    'Row': idx + 2,
                    'Study_Variable': study_var_name if study_var_name else 'Unknown',
                    'Study_Form': study_form_name,
                    'Study_Text': study_text_raw,
                    'Closest_HEAL_CDE_Concept': best['cde_name'],
                    'Concept_Score': best['concept_score'],
                    'Concept_CRF': best['crf_name'],
                    'Concept_Status': classify_concept_score(best['concept_score']),
                    'Encoding_Fidelity_Score': best['fidelity_score'],
                    'Encoding_Fidelity_Status': classify_fidelity_score(best['fidelity_score']),
                    'Expected_CRF': expected_raw,
                    'Expected_CRF_Mapped_To_KB': expected_kb,
                    'Protected_Family_Note': ''
                })

        # ============================================================
        # CASE 2: protected family row has only blocked cross-family proxies
        #         -> do NOT write a primary best match
        # ============================================================
        elif protected_mode and blocked_primary_candidates:
            blocked = blocked_primary_candidates[0]

            # Leave primary/best-match columns blank to avoid false exact assignment
            study_df.at[idx, 'Closest HEAL CDE Concept'] = None
            study_df.at[idx, 'Concept Match Score'] = None
            study_df.at[idx, 'Concept Match CRF'] = None
            study_df.at[idx, 'Encoding Fidelity Score'] = None
            study_df.at[idx, 'Encoding Fidelity Status'] = None

            study_df.at[idx, 'Best Match CDE Name'] = None
            study_df.at[idx, 'Best Match Score'] = None
            study_df.at[idx, 'Best Match CRF Name'] = None
            study_df.at[idx, 'Best Match Source'] = "Protected-family proxy blocked"

            study_df.at[idx, 'Potential Match 2 - CDE Name'] = None
            study_df.at[idx, 'Potential Match 2 - Score'] = None
            study_df.at[idx, 'Potential Match 2 - CRF Name'] = None
            study_df.at[idx, 'Potential Match 3 - CDE Name'] = None
            study_df.at[idx, 'Potential Match 3 - Score'] = None
            study_df.at[idx, 'Potential Match 3 - CRF Name'] = None

            # Store blocked proxy for human review
            study_df.at[idx, 'Blocked Primary Candidate'] = blocked['cde_name']
            study_df.at[idx, 'Blocked Primary Candidate Score'] = blocked['concept_score']
            study_df.at[idx, 'Blocked Primary Candidate CRF'] = blocked['crf_name']
            study_df.at[idx, 'Blocked Primary Candidate Fidelity'] = blocked['fidelity_score']
            study_df.at[idx, 'Protected Family Note'] = (
                f"Expected CRF '{expected_kb}' is protected. "
                f"Cross-family proxy '{blocked['cde_name']}' from '{blocked['crf_name']}' was blocked from becoming the primary best match."
            )

            study_df.at[idx, 'Concept Match Status'] = "Protected-family review needed"

            study_text_raw = f"{study_var_name} | {study_field_label} | {study_encoding}".strip()
            if len(study_text_raw) > 140:
                study_text_raw = study_text_raw[:137] + "..."

            low_confidence_matches.append({
                'Row': idx + 2,
                'Study_Variable': study_var_name if study_var_name else 'Unknown',
                'Study_Form': study_form_name,
                'Study_Text': study_text_raw,
                'Closest_HEAL_CDE_Concept': '',
                'Concept_Score': '',
                'Concept_CRF': '',
                'Concept_Status': 'Protected-family review needed',
                'Encoding_Fidelity_Score': '',
                'Encoding_Fidelity_Status': '',
                'Expected_CRF': expected_raw,
                'Expected_CRF_Mapped_To_KB': expected_kb,
                'Protected_Family_Note': (
                    f"Blocked proxy candidate: {blocked['cde_name']} ({blocked['crf_name']}) | "
                    f"Concept={blocked['concept_score']} | Fidelity={blocked['fidelity_score']}"
                )
            })

        else:
            continue

    # ----------------------------
    # Final output + duplicate audit
    # ----------------------------
    final_df = study_df.copy()

    # Add duplicate-audit review columns
    final_df = add_duplicate_audit_columns(final_df, study_var_col=var_col)

    low_conf_df = pd.DataFrame(low_confidence_matches) if low_confidence_matches else pd.DataFrame()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    study_basename = os.path.splitext(os.path.basename(study_file))[0]
    output_file = os.path.join(OUTPUT_DIR, f"{study_basename}_vlmd_conceptsplit_{timestamp}.xlsx")

    print("💾 Saving results...")
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        final_df.to_excel(writer, sheet_name="VLMD_Results", index=False)

        if not low_conf_df.empty:
            low_conf_df.to_excel(writer, sheet_name="Low_Confidence_Analysis", index=False)
            print(f"📝 {len(low_conf_df)} lower-confidence concept matches saved for analysis")
        else:
            print("📝 No Low_Confidence_Analysis sheet created.")

    print(f"💾 CDE matching complete. Results saved to {output_file}")
    print_matching_summary(final_df)

    return output_file


def apply_color_coding(output_file):
    """
    Apply color coding based on FINAL display status:
    - Green = High concept match
    - Orange = Possible concept match
    - Red = Weak concept match
    """
    print("\n🎨 Applying color coding and confidence levels...")

    wb = load_workbook(output_file)
    ws = wb['VLMD_Results'] if 'VLMD_Results' in wb.sheetnames else wb.active

    headers = [cell.value for cell in ws[1]]

    def find_col(header_name):
        for i, v in enumerate(headers, start=1):
            if isinstance(v, str) and v.strip() == header_name:
                return i
        return None

    status_col = find_col("Final Concept Match Status")
    if not status_col:
        status_col = find_col("Concept Match Status")

    score_col = find_col("Final Concept Match Score")
    if not score_col:
        score_col = find_col("Concept Match Score")
    if not score_col:
        score_col = find_col("Best Match Score")

    if not status_col and not score_col:
        print("⚠️ Could not find status or score columns. Skipping color coding.")
        wb.save(output_file)
        return

    confidence_col_idx = find_col("Confidence Level")
    if confidence_col_idx is None:
        confidence_col_idx = (status_col or score_col) + 1
        ws.insert_cols(confidence_col_idx)
        ws.cell(row=1, column=confidence_col_idx).value = "Confidence Level"

    FILL_GREEN  = PatternFill(fill_type="solid", fgColor="FFC6EFCE")
    FILL_ORANGE = PatternFill(fill_type="solid", fgColor="FFFFEB9C")
    FILL_RED    = PatternFill(fill_type="solid", fgColor="FFFFC7CE")

    for row_idx in range(2, ws.max_row + 1):
        confidence_cell = ws.cell(row=row_idx, column=confidence_col_idx)

        status_value = None
        if status_col:
            status_value = ws.cell(row=row_idx, column=status_col).value

        if isinstance(status_value, str) and status_value.strip():
            status_clean = status_value.strip()

            if status_clean == "High concept match":
                confidence_cell.value = status_clean
                if score_col:
                    ws.cell(row=row_idx, column=score_col).fill = FILL_GREEN

            elif status_clean == "Possible concept match":
                confidence_cell.value = status_clean
                if score_col:
                    ws.cell(row=row_idx, column=score_col).fill = FILL_ORANGE

            elif status_clean == "Weak concept match":
                confidence_cell.value = status_clean
                if score_col:
                    ws.cell(row=row_idx, column=score_col).fill = FILL_RED

            else:
                confidence_cell.value = status_clean

            continue

        # Fallback only if no status column is available
        if score_col:
            score = ws.cell(row=row_idx, column=score_col).value
            if score is None or score == "":
                continue

            try:
                score_num = float(score)
            except Exception:
                confidence_cell.value = "Unparseable score"
                continue

            score_cell = ws.cell(row=row_idx, column=score_col)

            if score_num >= CONCEPT_THRESHOLDS['high']:
                confidence_cell.value = "High concept match"
                score_cell.fill = FILL_GREEN
            elif score_num >= CONCEPT_THRESHOLDS['medium']:
                confidence_cell.value = "Possible concept match"
                score_cell.fill = FILL_ORANGE
            else:
                confidence_cell.value = "Weak concept match"
                score_cell.fill = FILL_RED

    wb.save(output_file)
    print("✨ Color coding applied successfully!")


print("✅ Updated compare_encodings() loaded.")
print("   Winner is now chosen by concept score.")
print("   Encoding is now reported separately as implementation fidelity.")
print("   Clearer FINAL output column names are now written.")
print("   Selective family-mode concepts are now supported in Stage 1.")

✅ Updated compare_encodings() loaded.
   Winner is now chosen by concept score.
   Encoding is now reported separately as implementation fidelity.
   Clearer FINAL output column names are now written.
   Selective family-mode concepts are now supported in Stage 1.


In [9]:
# ============================================================
# DUPLICATE AUDIT HELPERS
# Add this as a NEW cell below the override cell
# and above main(dry_run=False)
# ============================================================

def _safe_num(value, default=-1):
    """Convert values to float safely for tie-breaking."""
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default

def _looks_like_other_or_specify(var_name):
    """
    Lightweight detector for companion fields like:
    - other
    - specify
    - oth
    - txt / text
    """
    if pd.isna(var_name) or var_name is None:
        return False

    raw = str(var_name).strip().lower()

    patterns = [
        "other",
        "specify",
        "oth",
        "txt",
        "text"
    ]

    return any(p in raw for p in patterns)

def add_duplicate_audit_columns(final_df, study_var_col=None):
    """
    Post-processing duplicate audit.

    Groups by:
    - Final HEAL CDE Concept Match
    - Final Concept Match CRF

    Adds:
    - Duplicate Final Concept Count
    - Duplicate Group ID
    - Duplicate Group Members
    - Primary Representative for Final Concept
    - Duplicate Review Flag
    """
    audited_df = final_df.copy()

    audit_cols = [
        "Duplicate Final Concept Count",
        "Duplicate Group ID",
        "Duplicate Group Members",
        "Primary Representative for Final Concept",
        "Duplicate Review Flag"
    ]

    for col in audit_cols:
        audited_df[col] = None

    # only audit rows that actually have a final concept match
    valid_mask = (
        audited_df["Final HEAL CDE Concept Match"].notna() &
        audited_df["Final Concept Match CRF"].notna()
    )

    valid_df = audited_df[valid_mask].copy()

    if valid_df.empty:
        print("🧩 No rows available for duplicate audit.")
        return audited_df

    grouped = valid_df.groupby(
        ["Final HEAL CDE Concept Match", "Final Concept Match CRF"],
        dropna=False
    )

    review_group_count = 0

    for (concept_name, concept_crf), grp in grouped:
        idxs = grp.index.tolist()
        dup_count = len(idxs)

        group_id = f"{concept_crf} | {concept_name}"

        # Build readable member list
        if study_var_col and study_var_col in audited_df.columns:
            member_names = [
                str(audited_df.at[i, study_var_col]).strip()
                for i in idxs
                if pd.notna(audited_df.at[i, study_var_col])
            ]
        else:
            member_names = [f"row_{i}" for i in idxs]

        member_names = [m for m in member_names if m]
        member_string = ", ".join(member_names)

        # Pick primary representative:
        # 1) highest final concept score
        # 2) highest final encoding fidelity score
        # 3) lowest row index as stable tie-breaker
        primary_idx = sorted(
            idxs,
            key=lambda i: (
                _safe_num(audited_df.at[i, "Final Concept Match Score"]),
                _safe_num(audited_df.at[i, "Final Encoding Fidelity Score"]),
                -i
            ),
            reverse=True
        )[0]

        # Determine group-level review flag
        group_modes = set(
            str(v).strip().lower()
            for v in grp["Final Concept Match Mode"].dropna().tolist()
        )

        has_family_mode = "family" in group_modes

        has_other_specify = False
        if study_var_col and study_var_col in audited_df.columns:
            has_other_specify = any(
                _looks_like_other_or_specify(audited_df.at[i, study_var_col])
                for i in idxs
            )

        if dup_count == 1:
            group_flag = "Unique final concept match"
        elif has_family_mode:
            group_flag = "Allowed family concept repetition"
        elif has_other_specify:
            group_flag = "Review duplicate group (contains other/specify companion)"
            review_group_count += 1
        else:
            group_flag = "Review duplicate final concept match"
            review_group_count += 1

        # Write group results back to all rows in the group
        for i in idxs:
            audited_df.at[i, "Duplicate Final Concept Count"] = dup_count
            audited_df.at[i, "Duplicate Group ID"] = group_id
            audited_df.at[i, "Duplicate Group Members"] = member_string
            audited_df.at[i, "Primary Representative for Final Concept"] = (
                "Yes" if i == primary_idx else "No"
            )
            audited_df.at[i, "Duplicate Review Flag"] = group_flag

    print("✅ Duplicate audit columns added.")
    print(f"   Audited concept groups: {len(grouped)}")
    print(f"   Groups flagged for review: {review_group_count}")

    return audited_df

In [10]:
# ============================================================
# STAGE A RETRIEVAL POOLS
# Add this as a NEW cell above the override compare_encodings() cell
# ============================================================

def get_stage1_retrieval_pools(stage1_cde_df, has_anchor, expected_kb):
    """
    Build retrieval pools for Stage A.

    Behavior:
    - If the row has a usable CRF anchor and we can map it to a KB CRF,
      search that CRF first.
    - If nothing useful is found there, fall back to the full Stage 1 KB.
    """
    pools = []

    if has_anchor and expected_kb:
        anchored_df = stage1_cde_df[
            stage1_cde_df["CRF Name"].astype(str).str.strip() == str(expected_kb).strip()
        ].copy()

        if not anchored_df.empty:
            pools.append(("anchored_crf_first", anchored_df))

    # Always keep global fallback
    pools.append(("global_fallback", stage1_cde_df))

    return pools


print("✅ Stage A retrieval pools loaded.")
print("   - anchored rows search matched CRF first")
print("   - global fallback remains available")

✅ Stage A retrieval pools loaded.
   - anchored rows search matched CRF first
   - global fallback remains available


In [11]:
# ============================================================
# OVERRIDE SUMMARY + MATCHING LOGIC
# Add this as a NEW cell below the main() definition cell
# and above the final `main(dry_run=False)` execution cell.
# ============================================================

# Compatibility aliases so the old main()/color-coding logic doesn't choke
CONFIDENCE_THRESHOLDS = CONCEPT_THRESHOLDS
RESTRICT_TO_EXPECTED_CRF_FIRST = PREFER_EXPECTED_CRF_FOR_CONCEPT
ALLOW_GLOBAL_FALLBACK_FOR_BEST_MATCH = True

def print_matching_summary(final_df):
    """Updated summary: concept retrieval first, fidelity second."""
    processed_df = final_df[final_df['Closest HEAL CDE Concept'].notna()]
    total_processed = len(processed_df)

    if total_processed == 0:
        print("📊 No concept matches found.")
        return

    high_concept = len(processed_df[processed_df['Concept Match Score'] >= CONCEPT_THRESHOLDS['high']])
    medium_concept = len(processed_df[
        (processed_df['Concept Match Score'] >= CONCEPT_THRESHOLDS['medium']) &
        (processed_df['Concept Match Score'] < CONCEPT_THRESHOLDS['high'])
    ])
    low_concept = len(processed_df[processed_df['Concept Match Score'] < CONCEPT_THRESHOLDS['medium']])

    high_fidelity = len(processed_df[processed_df['Encoding Fidelity Score'] >= FIDELITY_THRESHOLDS['high']])
    medium_fidelity = len(processed_df[
        (processed_df['Encoding Fidelity Score'] >= FIDELITY_THRESHOLDS['medium']) &
        (processed_df['Encoding Fidelity Score'] < FIDELITY_THRESHOLDS['high'])
    ])
    low_fidelity = len(processed_df[processed_df['Encoding Fidelity Score'] < FIDELITY_THRESHOLDS['medium']])

    print(f"\n📊 Matching Summary:")
    print(f"   Total variables with a closest concept: {total_processed}")
    print(f"   🧠 High concept matches (≥{CONCEPT_THRESHOLDS['high']}): {high_concept} ({high_concept/total_processed*100:.1f}%)")
    print(f"   🧠 Possible concept matches ({CONCEPT_THRESHOLDS['medium']}-{CONCEPT_THRESHOLDS['high']-1}): {medium_concept} ({medium_concept/total_processed*100:.1f}%)")
    print(f"   🧠 Weak concept matches (<{CONCEPT_THRESHOLDS['medium']}): {low_concept} ({low_concept/total_processed*100:.1f}%)")

    print(f"\n   🛠️ High implementation fidelity (≥{FIDELITY_THRESHOLDS['high']}): {high_fidelity} ({high_fidelity/total_processed*100:.1f}%)")
    print(f"   🛠️ Medium implementation fidelity ({FIDELITY_THRESHOLDS['medium']}-{FIDELITY_THRESHOLDS['high']-1}): {medium_fidelity} ({medium_fidelity/total_processed*100:.1f}%)")
    print(f"   🛠️ Low implementation fidelity (<{FIDELITY_THRESHOLDS['medium']}): {low_fidelity} ({low_fidelity/total_processed*100:.1f}%)")


def compare_encodings(
    study_file,
    encoding_column='encodings',
    field_label_column='field_label',
    cde_file='./KnowledgeBase/Compiled_CORE_CDEs list_English_one sheet_as of 2025-01-28.xlsx',
    study_sheet='Sheet1',
    variable_name_column='name',
    dry_run=False
):
    """
    New behavior:
    - Select the winner by CONCEPT score
    - Describe implementation using ENCODING FIDELITY score
    - Keep old Best Match columns for compatibility with downstream steps
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print("📁 Output directory (absolute):", os.path.abspath(OUTPUT_DIR))

    # Load study data
    print("📂 Loading study data...")
    if study_file.endswith('.xlsx'):
        full_study_df = pd.read_excel(study_file, sheet_name=study_sheet)
    else:
        full_study_df = pd.read_csv(study_file)

    study_df = full_study_df.copy()

    if EXPECTED_CRF_COL in study_df.columns:
        no_crf_count = (study_df[EXPECTED_CRF_COL] == 'No CRF match').sum()
        print(f"✅ Processing all {len(study_df)} rows, including {no_crf_count} rows with 'No CRF match'.")
    else:
        print(f"✅ Processing all {len(study_df)} rows (no '{EXPECTED_CRF_COL}' column found).")

    if dry_run:
        print(f"🔍 DRY RUN: Would process {len(study_df)} variables against HEAL CDEs")
        return None

    # Detect columns robustly
    candidate_var_cols = [
        variable_name_column, 'Variable / Field Name', 'Variable/Field Name',
        'field_name', 'Field Name', 'variable_name', 'name', 'variable'
    ]
    var_col = next((c for c in candidate_var_cols if c in study_df.columns), None)

    candidate_form_cols = [
        FORM_NAME_COLUMN, 'Form Name', 'form_name', 'Form', 'form'
    ]
    form_col = next((c for c in candidate_form_cols if c in study_df.columns), None)

    if var_col:
        print(f"🔎 Using '{var_col}' as the study variable-name column.")
    else:
        print("⚠️ Could not find a variable-name column.")

    if form_col:
        print(f"🧾 Using '{form_col}' as the study form-name column.")
    else:
        print("⚠️ Could not find a form-name column. Form context score will be weaker.")

    # Initialize output columns
    new_cols = [
        # new clean-workflow columns
        'Closest HEAL CDE Concept',
        'Concept Match Score',
        'Concept Match CRF',
        'Concept Match Status',
        'Encoding Fidelity Score',
        'Encoding Fidelity Status',

        # legacy compatibility columns
        'Best Match CDE Name',
        'Best Match Score',
        'Best Match CRF Name',
        'Best Match Source',

        # top concept alternates
        'Potential Match 2 - CDE Name',
        'Potential Match 2 - Score',
        'Potential Match 2 - CRF Name',
        'Potential Match 3 - CDE Name',
        'Potential Match 3 - Score',
        'Potential Match 3 - CRF Name'
    ]

    for col in new_cols:
        study_df[col] = None

    # Normalize study fields
    print("🔧 Normalizing study variables...")
    study_df['Normalized Text'] = study_df[field_label_column].apply(normalize_string)
    study_df['Normalized Encoding'] = study_df[encoding_column].apply(normalize_string)

    if var_col:
        study_df['Normalized Variable Name'] = study_df[var_col].apply(normalize_string)
    else:
        study_df['Normalized Variable Name'] = ""

    if form_col:
        study_df['Normalized Form Name'] = study_df[form_col].apply(normalize_string)
    else:
        study_df['Normalized Form Name'] = ""

    study_df['Normalized Combined'] = study_df.apply(
        lambda row: normalize_string(
            f"{row.get(var_col, '')} | {row.get(field_label_column, '')} | {row.get(encoding_column, '')}"
        ),
        axis=1
    )

    # Load KB
    print("📚 Loading HEAL CDE database...")
    cde_df = pd.read_excel(cde_file, sheet_name='ALL')

    # Keep rows that at least have useful concept fields
    cde_df = cde_df.dropna(subset=['Variable Name', 'Additional Notes (Question Text)'], how='all').copy()

    cde_df["CRF Name"] = cde_df["CRF Name"].astype(str).str.strip()
    kb_crf_names = sorted(cde_df["CRF Name"].dropna().unique())
    map_crf = build_crf_mapper(kb_crf_names)

    cde_df['Normalized CDE Variable Name'] = cde_df['Variable Name'].apply(normalize_string)
    cde_df['Normalized Text'] = cde_df['Additional Notes (Question Text)'].apply(normalize_string)
    cde_df['Normalized Encoding'] = cde_df['PV Description'].apply(normalize_string)

    print(f"✅ Loaded {len(cde_df)} HEAL CDEs for comparison.")

    # Track concept matches that may need review
    low_confidence_matches = []

    print("🔍 Running concept retrieval first, then encoding fidelity scoring...")
    for idx, row in tqdm(study_df.iterrows(), total=len(study_df), desc="Matching variables", unit="vars"):

        # Skip rows with basically no usable text at all
        if not row.get('Normalized Variable Name', '') and not row.get('Normalized Text', ''):
            continue

        expected_raw = row.get(EXPECTED_CRF_COL, None)
        if pd.notna(expected_raw) and str(expected_raw).strip() and str(expected_raw).strip() != 'No CRF match':
            expected_kb = map_crf(expected_raw)
        else:
            expected_kb = None

        study_var_name = row.get(var_col, '') if var_col else ''
        study_field_label = row.get(field_label_column, '')
        study_form_name = row.get(form_col, '') if form_col else ''
        study_encoding = row.get(encoding_column, '')

        candidates = []

        for _, cde_row in cde_df.iterrows():
            cde_var = cde_row.get('Variable Name', '')
            cde_text = cde_row.get('Additional Notes (Question Text)', '')
            cde_encoding = cde_row.get('PV Description', '')
            cde_crf = cde_row.get('CRF Name', '')

            concept_score = concept_similarity_score(
                study_var_name=study_var_name,
                study_field_label=study_field_label,
                study_form_name=study_form_name,
                expected_crf=expected_kb,
                cde_var_name=cde_var,
                cde_question_text=cde_text,
                cde_crf_name=cde_crf
            )

            if concept_score < CONCEPT_THRESHOLDS['minimum_score']:
                continue

            fidelity_score = encoding_fidelity_score(study_encoding, cde_encoding)

            candidates.append({
                'cde_name': cde_var,
                'concept_score': round(concept_score, 1),
                'crf_name': cde_crf,
                'fidelity_score': round(fidelity_score, 1),
                'preferred_crf_hit': (expected_kb is not None and cde_crf == expected_kb)
            })

        if not candidates:
            continue

        # Sort by concept score first, then prefer expected CRF, then fidelity
        candidates = sorted(
            candidates,
            key=lambda x: (
                x['concept_score'],
                1 if x['preferred_crf_hit'] else 0,
                x['fidelity_score']
            ),
            reverse=True
        )

        # De-dupe by CDE name
        unique_candidates = []
        seen = set()
        for cand in candidates:
            if cand['cde_name'] in seen:
                continue
            unique_candidates.append(cand)
            seen.add(cand['cde_name'])

        top_candidates = unique_candidates[:TOP_N_CONCEPT_CANDIDATES]
        best = top_candidates[0]

        # New clean columns
        study_df.at[idx, 'Closest HEAL CDE Concept'] = best['cde_name']
        study_df.at[idx, 'Concept Match Score'] = best['concept_score']
        study_df.at[idx, 'Concept Match CRF'] = best['crf_name']
        study_df.at[idx, 'Concept Match Status'] = classify_concept_score(best['concept_score'])
        study_df.at[idx, 'Encoding Fidelity Score'] = best['fidelity_score']
        study_df.at[idx, 'Encoding Fidelity Status'] = classify_fidelity_score(best['fidelity_score'])

        # Legacy compatibility columns
        study_df.at[idx, 'Best Match CDE Name'] = best['cde_name']
        study_df.at[idx, 'Best Match Score'] = best['concept_score']
        study_df.at[idx, 'Best Match CRF Name'] = best['crf_name']
        study_df.at[idx, 'Best Match Source'] = "Concept retrieval"

        # Top alternates
        if len(top_candidates) > 1:
            study_df.at[idx, 'Potential Match 2 - CDE Name'] = top_candidates[1]['cde_name']
            study_df.at[idx, 'Potential Match 2 - Score'] = top_candidates[1]['concept_score']
            study_df.at[idx, 'Potential Match 2 - CRF Name'] = top_candidates[1]['crf_name']

        if len(top_candidates) > 2:
            study_df.at[idx, 'Potential Match 3 - CDE Name'] = top_candidates[2]['cde_name']
            study_df.at[idx, 'Potential Match 3 - Score'] = top_candidates[2]['concept_score']
            study_df.at[idx, 'Potential Match 3 - CRF Name'] = top_candidates[2]['crf_name']

        # Review sheet now tracks weaker concept matches
        if best['concept_score'] < CONCEPT_THRESHOLDS['high']:
            study_text_raw = f"{study_var_name} | {study_field_label} | {study_encoding}".strip()
            if len(study_text_raw) > 140:
                study_text_raw = study_text_raw[:137] + "..."

            low_confidence_matches.append({
                'Row': idx + 2,
                'Study_Variable': study_var_name if study_var_name else 'Unknown',
                'Study_Form': study_form_name,
                'Study_Text': study_text_raw,
                'Closest_HEAL_CDE_Concept': best['cde_name'],
                'Concept_Score': best['concept_score'],
                'Concept_CRF': best['crf_name'],
                'Concept_Status': classify_concept_score(best['concept_score']),
                'Encoding_Fidelity_Score': best['fidelity_score'],
                'Encoding_Fidelity_Status': classify_fidelity_score(best['fidelity_score']),
                'Expected_CRF': expected_raw,
                'Expected_CRF_Mapped_To_KB': expected_kb
            })

    # Final output
    final_df = study_df.copy()
    low_conf_df = pd.DataFrame(low_confidence_matches) if low_confidence_matches else pd.DataFrame()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    study_basename = os.path.splitext(os.path.basename(study_file))[0]
    output_file = os.path.join(OUTPUT_DIR, f"{study_basename}_vlmd_conceptsplit_{timestamp}.xlsx")

    print("💾 Saving results...")
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        final_df.to_excel(writer, sheet_name="VLMD_Results", index=False)

        if not low_conf_df.empty:
            low_conf_df.to_excel(writer, sheet_name="Low_Confidence_Analysis", index=False)
            print(f"📝 {len(low_conf_df)} lower-confidence concept matches saved for analysis")
        else:
            print("📝 No Low_Confidence_Analysis sheet created.")

    print(f"💾 CDE matching complete. Results saved to {output_file}")
    print_matching_summary(final_df)

    return output_file

print("✅ Updated compare_encodings() loaded.")
print("   Winner is now chosen by concept score.")
print("   Encoding is now reported separately as implementation fidelity.")

✅ Updated compare_encodings() loaded.
   Winner is now chosen by concept score.
   Encoding is now reported separately as implementation fidelity.


In [12]:
main(dry_run=False)

🚀 Starting HEAL CDE Variable Level Metadata Matching
📋 Concept retrieval thresholds: High ≥75%, Possible ≥50%, Minimum ≥25%
📋 Encoding fidelity thresholds: High ≥80%, Medium ≥50%

🧭 Concept retrieval settings:
   Expected CRF column        : HEAL Core CRF Match
   Prefer expected CRF        : True
   CRF mapping threshold      : 85
   CRF concept bonus          : 8
   (Potential Match 2/3 are alternate concept candidates.)
📁 Output directory (absolute): c:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out
📂 Loading study data...
✅ Processing all 182 rows, including 97 rows with 'No CRF match'.
🔎 Using 'Variable / Field Name' as the study variable-name column.
🧾 Using 'Form Name' as the study form-name column.
🔧 Normalizing study variables...
📚 Loading HEAL CDE database...
✅ Loaded 425 HEAL CDEs for comparison.
🔍 Running concept retrieval first, then encoding fidelity scoring...


Matching variables: 100%|██████████| 182/182 [00:08<00:00, 20.77vars/s]


💾 Saving results...
📝 131 lower-confidence concept matches saved for analysis
💾 CDE matching complete. Results saved to c:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058.xlsx

📊 Matching Summary:
   Total variables with a closest concept: 182
   🧠 High concept matches (≥75): 51 (28.0%)
   🧠 Possible concept matches (50-74): 38 (20.9%)
   🧠 Weak concept matches (<50): 93 (51.1%)

   🛠️ High implementation fidelity (≥80): 36 (19.8%)
   🛠️ Medium implementation fidelity (50-79): 46 (25.3%)
   🛠️ Low implementation fidelity (<50): 100 (54.9%)

✅ Output file created:
   📄 c:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_conceptsplit_20260428_232058.xlsx

🎨 Applying color coding and confidence levels...
✨ Color coding applied successfully!

🎉 VLMD CDE matching complete!
📊 Results saved to: c:\Users\lmaefos\Code S

# HEAL CDE Fuzzy Matching Script

This script automates the matching of study data dictionaries to the HEAL Core Common Data Elements (CDEs), using smart fuzzy text comparison and a color-coded Excel output for easier review.

It is designed to help identify the **Best Match** and **Top 2 Potential Matches** for each study variable — even if the wording or encoding order is slightly different from the CDE standard.

---

## ✨ Key Features
- **Fuzzy Matching**: Uses token-based similarity (`Token Set Ratio`) to handle small typos and different word orders.
- **Normalized Comparisons**: Cleans and standardizes text for reliable matching.
- **Separate Output Folder**: All results are saved neatly into an `/out/` subfolder.
- **Color Coded Scores**:  
  - 🟩 **Green** for matches ≥ 80%  
  - 🟧 **Orange** for matches 51–79%  
  - 🟥 **Red** for matches ≤ 50%
- **No Duplicate Matches**: Ensures Best Match and Potential Matches are truly different.

---

## 🚀 How It Works

1. **Input**:
   - A study data dictionary (Excel `.xlsx` or CSV `.csv` file).
   - The master HEAL CDEs file (Excel file).

2. **Process**:
   - Normalize (clean) text by lowercasing, removing special characters, and preserving logical structures like equal signs.
   - Compare the study's "Encoding + Field Label" to the CDE's "PV Description + Question Text".
   - Select the best match and top two alternatives based on fuzzy matching scores.
   - Color-code the best match scores for easy visualization.

3. **Output**:
   - A new Excel file saved into `/out/`, with best matches listed and scores color-coded.

---

## 🛠️ Requirements

Install the following Python packages:

```bash
pip install pandas openpyxl fuzzywuzzy
```

---

## 📂 Folder Structure

```
/in/         # Input study files
/out/        # Output matched files (automatically created if not present)
/KnowledgeBase/ # Contains the HEAL Core CDE file
script.py    # Your main script
```

---

## 📋 Usage Example

```bash
python script.py
```

The output file will appear in the `/out/` folder and will be named something like:

```
SAMPLE_sprint_2020-12-16_vlmd_cdesearch.xlsx
```

---

## 🧠 Notes

- The script **requires** the correct columns to be named in the study file (e.g., `Choices, Calculations, OR Slider Labels` and `Field Label`).
- Only the **Best Match Score** column is color coded for quick review.
- Ensure your HEAL CDE master file contains the necessary columns: `PV Description` and `Additional Notes (Question Text)`.


# Notes for improvement
- copyrighted CDEs don't populate a 'Best Match CDE Name' because the knowledge base is empty for fields in the permissible values for copyrighted CDEs. 
- top 3 matches populate, sometimes there's no "Best Match", but there is a value for "Potential Matches" (the best match is missing)
- CRF level matches appear to work well
- VLMD matches do not appear to work well
- What if i add an additional assistant that will review each row, read through the following:
    - original variable name, original form name, original description, canonical CRF name rationale, HEAL Core CRF Match rationale, and identify the best Form name and CDE match 
